<a href="https://colab.research.google.com/github/amjsebas/proyecto-final-IA-finanzas-aerolineas-2/blob/main/notebooks/analisis_exploratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema Multi-Factor de Senales de Trading con LLMs

**Curso:** Modelos de Inteligencia Artificial para Finanzas - EGADE Business School
**Profesor:** Luis Angel Lozano Medina
**Entrega:** 08 de junio de 2026
**Equipo:** Mauricio Jazo, Sebastian Aceves, Jose Hernandez

**Universo:** AAL, DAL, UAL, LUV, JBLU, ULCC | **Benchmark:** ETF JETS | **Horizonte:** 5 dias habiles

Notebook v3 - rebuildeado limpio. Cada celda es idempotente, los checkpoints viven en `/content/saved/` (sobrevive runtime restarts), y hay smoke test de Gemini ANTES del loop grande.

**Para correr:** Runtime -> Restart and Run All. Te pide PAT + Gemini key. Tarda ~25-30 min.

## Paso 0 - Setup

In [ ]:
# Pin combo conocido como compatible (sin RuntimeError de torchvision ni conflictos econml/numpy)
!pip install --quiet --upgrade yfinance finvizfinance ta google-generativeai
!pip install --quiet "numpy<2.1" econml --force-reinstall
!pip install --quiet torch==2.5.1 torchvision==0.20.1 transformers==4.46.0 --force-reinstall
print("Dependencias instaladas.")
print("IMPORTANTE: ve a 'Entorno de ejecucion -> Reiniciar sesion' y vuelve a correr todo.")

In [ ]:
import sys, os, shutil, warnings
from pathlib import Path
# Clone del repo (público — no requiere PAT)
REPO_URL = "https://github.com/amjsebas/proyecto-final-IA-finanzas-aerolineas-2.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR) or not os.path.exists(f"{REPO_DIR}/src"):
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    !git clone https://github.com/amjsebas/proyecto-final-IA-finanzas-aerolineas-2.git {REPO_DIR}
else:
    print(f"Repo ya existe en {REPO_DIR} - skip clone.")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

# Carpeta de checkpoints FUERA del repo (sobrevive re-clones)
SAVED_DIR = "/content/saved"
os.makedirs(SAVED_DIR, exist_ok=True)
print(f"Checkpoints dir: {SAVED_DIR}")

# Gemini API key
from getpass import getpass
GEMINI_API_KEY = getpass("Pega tu Gemini API key: ")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
print("API key configurada.")

In [ ]:
# Imports + sys.path
sys.path.insert(0, "/content/repo")
sys.path.insert(0, "/content/repo/src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

import data_collection, technical_indicators, custom_signals
import sentiment_analysis, signal_combiner
import causal_effects, causal_weighting
import backtesting, evaluation

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print("Modulos cargados OK.")

### Smoke test - VALIDA Gemini API ANTES del loop grande

Hace UNA call a Gemini con cada modelo candidato. Reporta cual funciona y usa ese para el resto del pipeline.

In [ ]:
import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)

GEMINI_CANDIDATES = [
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-1.5-flash-002",
    "gemini-2.0-flash",
]

GEMINI_MODEL = None
for candidate in GEMINI_CANDIDATES:
    try:
        m = genai.GenerativeModel(candidate)
        resp = m.generate_content("ping",
            generation_config={"temperature": 0.0, "response_mime_type": "application/json"})
        print(f"  OK - {candidate}")
        GEMINI_MODEL = candidate
        break
    except Exception as e:
        print(f"  {candidate}: {str(e)[:80]}")
        continue

if GEMINI_MODEL is None:
    raise RuntimeError("Ningun modelo de Gemini disponible. Verifica API key + billing.")
print(f"\nModelo activo para el pipeline: {GEMINI_MODEL}")

# Smoke test yfinance
import yfinance as yf
_test = yf.download("AAL", period="5d", progress=False)
print(f"\nyfinance OK - {len(_test)} dias de prueba bajados")

In [ ]:
# Config del proyecto
UNIVERSE   = ["AAL", "DAL", "UAL", "LUV", "JBLU", "ULCC"]
BENCHMARK  = "JETS"
HORIZON    = 5
INITIAL_CAPITAL = 100_000
DATA_START = "2024-06-01"
BUY_THRESHOLD  =  0.25
SELL_THRESHOLD = -0.25
CONFIDENCE_FLOOR = 0.60
MAX_POSITION_PCT = 0.20
AUG_CAP    = 250          # cap de augmentadas adicionales a las 43 seed
HALFLIFE_DAYS = 3

OUTPUTS_DIR = Path("/content/repo/outputs"); OUTPUTS_DIR.mkdir(exist_ok=True)
DATA_PROC   = Path("/content/repo/data/processed"); DATA_PROC.mkdir(parents=True, exist_ok=True)
print("Config OK.")

## Fase 2 - Recoleccion y augmentacion (cap 250 augmentadas)

In [6]:
# Seed CSV (43 headlines manuales)
df_seed = pd.read_csv("/content/repo/data/raw/airline_news_for_model.csv", parse_dates=["date"])
print(f"Seed: {len(df_seed)} headlines (use_in_model=yes: {(df_seed['use_in_model'].str.lower()=='yes').sum()})")

COMPANY_NAMES = {"AAL":"American Airlines","DAL":"Delta Airlines","UAL":"United Airlines",
                 "LUV":"Southwest Airlines","JBLU":"JetBlue","ULCC":"Frontier Airlines"}

# Augmentacion - cap finviz a ~45/ticker para que despues del dedup nos quede ~250
print("\nAumentando con yfinance...")
df_yf = data_collection.fetch_yfinance_news(UNIVERSE, COMPANY_NAMES)
print(f"  yfinance: {len(df_yf)} headlines")

print("Aumentando con finvizfinance (cap 45/ticker)...")
df_fv = data_collection.fetch_finviz_news(UNIVERSE, COMPANY_NAMES, max_per_ticker=45)
print(f"  finviz:   {len(df_fv)} headlines")

df_aug = data_collection.consolidate_news(df_yf, df_fv)
df_aug = df_aug[~df_aug["headline"].isin(df_seed["headline"])].reset_index(drop=True)
df_aug = df_aug.head(AUG_CAP)
print(f"\nAugmentadas (cap {AUG_CAP}): {len(df_aug)}")

Seed: 60 headlines (use_in_model=yes: 43)

Aumentando con yfinance...
  yfinance: 60 headlines
Aumentando con finvizfinance (cap 45/ticker)...


KeyboardInterrupt: 

## Fase 4a - Auto-clasificacion de las 250 augmentadas con Gemini

Con checkpoint en `/content/saved/`. Si la celda truena a la mitad, vuelves a correrla y RETOMA desde donde quedo (no re-procesa lo que ya tiene).

In [ ]:
# Idempotente: si ya tenemos df_aug_classified completo en memoria, skip
need_run = ("df_aug_classified" not in globals()) or (len(df_aug_classified) < len(df_aug) * 0.9)

if need_run:
    df_aug_classified = sentiment_analysis.auto_classify_unlabeled(
        df_aug,
        api_key=GEMINI_API_KEY,
        model_name=GEMINI_MODEL,
        confidence_floor=0.5,
        checkpoint_path=f"{SAVED_DIR}/gemini_autoclass_ckpt.csv",
        sleep_between=0.0,
    )
    df_aug_classified["date"] = pd.to_datetime(df_aug_classified["date"])
    # Backup persistente
    df_aug_classified.to_csv(f"{SAVED_DIR}/df_aug_classified.csv", index=False)
    print(f"\nGuardado en {SAVED_DIR}/df_aug_classified.csv")
else:
    print(f"df_aug_classified ya existe ({len(df_aug_classified)} filas) - skip.")

In [ ]:
# Concat seed + augmentadas, filtra al modelo
df_all = pd.concat([df_seed, df_aug_classified], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["ticker","headline"])

df_model = df_all[df_all["use_in_model"].astype(str).str.lower().str.strip() == "yes"].copy().reset_index(drop=True)
df_model["date"] = pd.to_datetime(df_model["date"])
print(f"Total headlines para el modelo: {len(df_model)}")
print(df_model["ticker"].value_counts())

## Fase 4b - Sentiment hibrido FinBERT + Gemini + comparacion

In [ ]:
# FIX: Drop columnas finbert_* y llm_* viejas si existen (cell idempotente)
old_cols = [c for c in df_model.columns if c.startswith("finbert_") or c.startswith("llm_")]
if old_cols:
    df_model = df_model.drop(columns=old_cols)
    print(f"Drop {len(old_cols)} columnas pre-existentes")

# FinBERT
print("Corriendo FinBERT (primera vez ~30s)...")
fb = sentiment_analysis.score_finbert(df_model["headline"].tolist())
df_model = pd.concat([df_model.reset_index(drop=True), fb], axis=1)
print(f"FinBERT OK - {len(df_model)} headlines")

# Gemini sobre los del modelo (otra vez con checkpoint por las dudas)
llm = sentiment_analysis.score_gemini(
    df_model, api_key=GEMINI_API_KEY, model_name=GEMINI_MODEL,
    sleep_between=0.0,
    checkpoint_path=f"{SAVED_DIR}/gemini_sentiment_ckpt.csv")
df_model = pd.concat([df_model.reset_index(drop=True), llm], axis=1)
print(f"Gemini sentiment OK")

# Hibrido
df_model = sentiment_analysis.hybrid_sentiment(df_model, llm_confidence_threshold=0.5)
print("\nFuente del sentiment final:")
print(df_model["sentiment_source"].value_counts())

In [ ]:
# Comparacion FinBERT vs Gemini
comp = sentiment_analysis.compare_finbert_vs_gemini(df_model)
if comp.get("available"):
    print(f"Pearson  : {comp['pearson']:+.3f}")
    print(f"Spearman : {comp['spearman']:+.3f}")
    print(f"Sign agree: {comp['sign_agreement_pct']:.1%}  (n={comp['n']})")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].scatter(df_model["finbert_sentiment"], df_model["llm_sentiment_score"], alpha=0.6, s=30)
    ax[0].plot([-1,1],[-1,1],"--",color="gray")
    ax[0].axhline(0,color="gray",lw=0.5); ax[0].axvline(0,color="gray",lw=0.5)
    ax[0].set_xlabel("FinBERT"); ax[0].set_ylabel("Gemini")
    ax[0].set_title(f"FinBERT vs Gemini (Spearman {comp['spearman']:+.3f})")
    ax[1].hist([df_model["finbert_sentiment"], df_model["llm_sentiment_score"]], bins=15,
               label=["FinBERT","Gemini"], alpha=0.7)
    ax[1].set_xlabel("Sentiment"); ax[1].set_title("Distribuciones"); ax[1].legend()
    plt.tight_layout(); plt.savefig(OUTPUTS_DIR/"llm_vs_finbert.png", dpi=120, bbox_inches="tight")
    plt.show()

## Fase 5a - Anti-leakage timestamp alignment + agregacion ticker-fecha

In [ ]:
def shift_to_next_business_day(d):
    return (pd.Timestamp(d) + pd.tseries.offsets.BDay(1)).normalize()

df_model["effective_date"] = df_model["date"].apply(shift_to_next_business_day)

def aggregate_ticker_date(g):
    n = len(g); s = g["sentiment_score"].values; c = g["confidence"].values
    days_old = (g["effective_date"] - g["date"]).dt.days.values
    w = np.exp(-days_old / HALFLIFE_DAYS)
    return pd.Series({
        "news_count": n,
        "mean_sentiment": s.mean() if n else 0,
        "positive_share": (s>0).mean() if n else 0,
        "negative_share": (s<0).mean() if n else 0,
        "confidence_weighted_sentiment": (s*c).sum()/(c.sum() if c.sum() else 1),
        "recency_weighted_sentiment": (s*w).sum()/(w.sum() if w.sum() else 1),
        "main_event_type": g["event_type_final"].astype(str).mode().iat[0] if n else "unknown",
    })

mask_ind = df_model["relevance_level"].astype(str).str.lower().isin(["direct","indirect"])
df_ind = df_model[mask_ind]
df_sec = df_model[df_model["relevance_level"].astype(str).str.lower() == "sector"].copy()

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    signals_news = df_ind.groupby(["ticker","effective_date"], as_index=False).apply(aggregate_ticker_date).reset_index(drop=True)

df_sec["effective_date"] = df_sec["date"].apply(shift_to_next_business_day)
sector = (df_sec.drop_duplicates(subset=["effective_date","headline"])
          .groupby("effective_date", as_index=False)
          .agg(sector_sentiment=("sentiment_score","mean"),
               sector_news_count=("sentiment_score","size")))

signals_news = signals_news.rename(columns={"effective_date":"date"})
sector = sector.rename(columns={"effective_date":"date"})
signals_news = signals_news.merge(sector, on="date", how="left")
signals_news["sector_sentiment"] = signals_news["sector_sentiment"].fillna(0)
signals_news["sector_news_count"] = signals_news["sector_news_count"].fillna(0).astype(int)
signals_news["relative_sentiment"] = signals_news["mean_sentiment"] - signals_news["sector_sentiment"]
signals_news["sentiment_signal"] = signals_news["mean_sentiment"]
print(f"Signals ticker-fecha: {len(signals_news)}")

## Fase 2/3 - Precios + indicadores tecnicos + senales propias

In [ ]:
# Precios
prices = data_collection.fetch_prices(UNIVERSE + [BENCHMARK], start=DATA_START)
print(f"Precios cargados: {list(prices.keys())}")

# Controles de precio
controls = data_collection.compute_price_controls(prices, BENCHMARK)

# Indicadores tecnicos
techs = []
for tk in UNIVERSE:
    ind = technical_indicators.compute_all_indicators(prices[tk])
    agg = technical_indicators.aggregate_by_category(ind)
    agg["ticker"] = tk; agg["date"] = agg.index
    techs.append(agg.reset_index(drop=True))
df_tech = pd.concat(techs, ignore_index=True)

# Senales propias
custs = []
bench_close = prices[BENCHMARK]["Close"]
for tk in UNIVERSE:
    cs = custom_signals.compute_custom_signals(prices[tk]["Close"], bench_close)
    cs["custom_signal"] = custom_signals.aggregate_custom_signal(cs)
    cs["ticker"] = tk; cs["date"] = cs.index
    custs.append(cs.reset_index(drop=True))
df_cust = pd.concat(custs, ignore_index=True)
print(f"Tecnicos: {df_tech.shape} | Customs: {df_cust.shape}")

## Fase 5 - Modeling table + signal combiner

In [ ]:
# Merge tecnicos + customs + controles + sentiment
df_features = df_tech.merge(df_cust[["ticker","date","custom_signal"]], on=["ticker","date"], how="outer")
df_features["date"] = pd.to_datetime(df_features["date"])
df_features = df_features.merge(controls, on=["ticker","date"], how="left")
df_features = df_features.merge(signals_news, on=["ticker","date"], how="left")
df_features["sentiment_signal"] = df_features["sentiment_signal"].fillna(0)

# Target: future excess return a 5d hábiles
px = pd.concat({tk: prices[tk]["Close"] for tk in UNIVERSE + [BENCHMARK]}, axis=1)
px.columns = px.columns.get_level_values(0)
fwd = px.shift(-HORIZON) / px - 1

def future_excess(row):
    if row["ticker"] not in fwd.columns or row["date"] not in fwd.index:
        return np.nan
    s = fwd.loc[row["date"], row["ticker"]]; b = fwd.loc[row["date"], BENCHMARK]
    return s - b

df_features["future_excess_return_5d"] = df_features.apply(future_excess, axis=1)
df_features["next_return"] = df_features.groupby("ticker")["past_5d_return"].shift(-1) / 5
df_features["outperform_label"] = np.where(df_features["future_excess_return_5d"].isna(), np.nan,
                                            (df_features["future_excess_return_5d"]>0).astype(float))

modeling_table = df_features.dropna(subset=["future_excess_return_5d"]).copy().reset_index(drop=True)
modeling_table.to_csv(DATA_PROC/"modeling_table.csv", index=False)
print(f"Modeling table: {modeling_table.shape}")
print(f"Tickers: {sorted(modeling_table.ticker.unique())}")
print(f"Date range: {modeling_table.date.min().date()} -> {modeling_table.date.max().date()}")

In [ ]:
# FIX: rebalancear pesos base para que sumen 1.0 (ADX queda como multiplicador, no aditivo)
signal_combiner.DEFAULT_WEIGHTS = {
    "sentiment_signal":  0.16, "momentum_signal":   0.42, "trend_signal":      0.10,
    "volatility_signal": 0.10, "volume_signal":     0.14, "custom_signal":     0.08,
}
signal_combiner.WEIGHT_CONFIGS["base"] = dict(signal_combiner.DEFAULT_WEIGHTS)

# Signal combiner - 4 configs de pesos
experiments = signal_combiner.run_weight_experiments(modeling_table,
    buy_threshold=BUY_THRESHOLD, sell_threshold=SELL_THRESHOLD)

print("Distribucion de senales por configuracion:")
for name, df_e in experiments.items():
    c = df_e["signal_discrete"].value_counts()
    print(f"  {name:18s} BUY {c.get('BUY',0):>4d} HOLD {c.get('HOLD',0):>4d} SELL {c.get('SELL',0):>4d}")

modeling_table = experiments["base"]

In [ ]:
# Reinstalar econml y forzar reimport del modulo causal_effects
!pip install econml --quiet --upgrade

import sys
for m in list(sys.modules):
    if m.startswith('causal_effects') or m.startswith('econml'):
        del sys.modules[m]
import causal_effects
print(f"ECONML disponible: {causal_effects.ECONML_AVAILABLE}")

In [ ]:
# Diagnostico: ver el error exacto al importar econml
try:
    import econml
    print(f"econml version: {econml.__version__}")
    from econml.dml import CausalForestDML
    print("CausalForestDML import OK")
except Exception as e:
    print(f"\nERROR al importar econml:")
    print(f"  Tipo: {type(e).__name__}")
    print(f"  Mensaje: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
import os
os.makedirs("/content/saved", exist_ok=True)
modeling_table.to_csv("/content/saved/modeling_table_full.csv", index=False)
print(f"Guardado: modeling_table ({len(modeling_table)} filas, {len(modeling_table.columns)} cols)")
print(f"Tickers: {sorted(modeling_table.ticker.unique())}")
print(f"Date range: {modeling_table.date.min()} a {modeling_table.date.max()}")

In [ ]:
!pip install "numpy<2.1" econml --quiet --force-reinstall
print("Hecho. Ahora: Entorno de ejecucion -> Reiniciar sesion")

In [ ]:
# Recuperar modeling_table + configuracion + modulos despues del restart
import sys, os
sys.path.insert(0, "/content/repo")
sys.path.insert(0, "/content/repo/src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

# Reload modeling_table desde disco
modeling_table = pd.read_csv("/content/saved/modeling_table_full.csv", parse_dates=["date"])
print(f"Cargados {len(modeling_table)} filas, {len(modeling_table.columns)} cols")

# Verificar econml ya funciona
import causal_effects
print(f"ECONML disponible: {causal_effects.ECONML_AVAILABLE}")

# Importar todos los modulos para el resto del pipeline
import signal_combiner, causal_weighting, backtesting, evaluation
print("Modulos cargados.")

# Re-definir constantes que usan las celdas de abajo
SIGNAL_COLS = ["sentiment_signal","momentum_signal","trend_signal",
               "volatility_signal","volume_signal","custom_signal"]
CONTEXT_COLS = ["atr_pct","adx_signal","volume_spike",
                "past_5d_return","past_20d_return","volatility_20d",
                "volume_change_20d","market_return_20d","news_count"]
BUY_THRESHOLD = 0.25
SELL_THRESHOLD = -0.25
HORIZON = 5
INITIAL_CAPITAL = 100_000
MAX_POSITION_PCT = 0.20
CONFIDENCE_FLOOR = 0.60

# Rebalanceo de pesos (mismo fix de antes)
signal_combiner.DEFAULT_WEIGHTS = {
    "sentiment_signal":  0.16, "momentum_signal":   0.42, "trend_signal":      0.10,
    "volatility_signal": 0.10, "volume_signal":     0.14, "custom_signal":     0.08,
}
signal_combiner.WEIGHT_CONFIGS["base"] = dict(signal_combiner.DEFAULT_WEIGHTS)

# Necesitamos prices para el backtest (sin tener que re-correr Fase 2/3)
import data_collection
UNIVERSE = ["AAL","DAL","UAL","LUV","JBLU","ULCC"]
BENCHMARK = "JETS"
DATA_START = "2024-06-01"
prices = data_collection.fetch_prices(UNIVERSE + [BENCHMARK], start=DATA_START)
print(f"Precios cargados: {list(prices.keys())}")

# Output dirs
from pathlib import Path
OUTPUTS_DIR = Path("/content/repo/outputs"); OUTPUTS_DIR.mkdir(exist_ok=True)
DATA_PROC   = Path("/content/repo/data/processed"); DATA_PROC.mkdir(parents=True, exist_ok=True)

# Necesitamos experiments[sentiment_heavy] para el backtest B
experiments = signal_combiner.run_weight_experiments(modeling_table,
    buy_threshold=BUY_THRESHOLD, sell_threshold=SELL_THRESHOLD)
modeling_table = experiments["base"]
print("Estado recuperado. Listo para Fase 5.5.")

## Fase 5.5 - Causal HTE (donde el profe quiere que brillemos)

In [ ]:
SIGNAL_COLS = ["sentiment_signal","momentum_signal","trend_signal",
               "volatility_signal","volume_signal","custom_signal"]
CONTEXT_COLS = ["atr_pct","adx_signal","volume_spike",
                "past_5d_return","past_20d_return","volatility_20d",
                "volume_change_20d","market_return_20d","news_count"]

for c in CONTEXT_COLS:
    if c not in modeling_table.columns:
        modeling_table[c] = 0.0

modeling_table["next_return"] = modeling_table["next_return"].fillna(
    modeling_table["future_excess_return_5d"] / 5)

cfg = causal_effects.CausalConfig(outcome_col="next_return", min_samples=20, min_samples_leaf=3)
est = causal_effects.CausalEffectsEstimator(
    signal_cols=SIGNAL_COLS, context_cols=CONTEXT_COLS, config=cfg)

try:
    est.fit(modeling_table)
    print("Causal model entrenado")
    ate_table = est.ate_summary()
    print("\nATE por senal:")
    print(ate_table.round(5))
    ate_table.to_csv(OUTPUTS_DIR/"hte_segments_table.csv", index=False)
except Exception as e:
    print(f"Causal fit fallo: {e}")
    ate_table = pd.DataFrame()
    est = None

In [ ]:
# Predict effects + causal weights + dynamic combine + heatmap
if est is not None:
    modeling_table = est.predict_effects(modeling_table)
    base_w = signal_combiner.DEFAULT_WEIGHTS
    modeling_table = causal_weighting.compute_causal_weights(modeling_table, SIGNAL_COLS, base_w, gamma=1.0)
    modeling_table = causal_weighting.combine_signals_with_causal_weights(modeling_table, SIGNAL_COLS,
        buy_threshold=BUY_THRESHOLD, sell_threshold=SELL_THRESHOLD)

    weight_cols = [f"causal_weight_{s}" for s in SIGNAL_COLS]
    mt = modeling_table.copy()
    mt["month"] = pd.to_datetime(mt["date"]).dt.to_period("M").astype(str)
    hm = mt.groupby("month")[weight_cols].mean().rename(columns=lambda c: c.replace("causal_weight_",""))
    fig, ax = plt.subplots(figsize=(10, 4.5))
    sns.heatmap(hm.T, cmap="RdBu_r", center=0, cbar_kws={"label":"Peso causal"}, ax=ax)
    ax.set_xlabel("Mes"); ax.set_ylabel("Senal")
    ax.set_title("Pesos causales dinamicos por mes")
    plt.tight_layout(); plt.savefig(OUTPUTS_DIR/"causal_weight_heatmap.png", dpi=120, bbox_inches="tight")
    plt.show()

## Fase 6 - Backtesting

In [ ]:
modeling_table["next_return"] = modeling_table["next_return"].fillna(modeling_table["future_excess_return_5d"]/5)

bt_base = backtesting.backtest_strategy(modeling_table, signal_col="signal_discrete", initial_capital=INITIAL_CAPITAL)
bt_sentiment_heavy = backtesting.backtest_strategy(experiments["sentiment_heavy"], signal_col="signal_discrete", initial_capital=INITIAL_CAPITAL)
if "signal_discrete_causal" in modeling_table.columns:
    bt_causal = backtesting.backtest_strategy(modeling_table, signal_col="signal_discrete_causal", initial_capital=INITIAL_CAPITAL)
else:
    bt_causal = bt_base.copy()
bh = backtesting.backtest_buy_and_hold(prices[BENCHMARK]["Close"], initial_capital=INITIAL_CAPITAL)

backtesting.plot_equity_curve(bt_base, bh, title="Equity Curve - Base vs Buy & Hold",
    out_path=str(OUTPUTS_DIR/"equity_curve.png"))
backtesting.plot_equity_curve(bt_causal, bh, title="Equity Curve - Causal vs Buy & Hold",
    out_path=str(OUTPUTS_DIR/"causal_vs_base_backtest.png"))

## Fase 7 - Clasificacion + ROC + KS

In [ ]:
gt = evaluation.make_ground_truth(modeling_table["next_return"], threshold_pct=1.0)
pred = modeling_table["signal_discrete"]

evaluation.plot_confusion_matrix(gt, pred, out_path=str(OUTPUTS_DIR/"confusion_matrix.png"))
metrics_cls = evaluation.classification_metrics(gt, pred)
print("Metricas:")
for k, v in metrics_cls.items():
    print(f"  {k:20s}: {v:.3f}")

y_bin = evaluation.binarize_movement(modeling_table["next_return"])
auc = evaluation.plot_roc(y_bin, modeling_table["signal_raw"], out_path=str(OUTPUTS_DIR/"roc_curve.png"))
ks  = evaluation.ks_statistic(y_bin, modeling_table["signal_raw"])
print(f"\nROC-AUC: {auc:.3f}\nKS: {ks:.3f}")

## Fase 8 - Riesgo: calibracion + sizing + stops

In [ ]:
hit = (modeling_table["signal_discrete"] == gt).astype(int)
cal = evaluation.plot_calibration(modeling_table["confidence"], hit, out_path=str(OUTPUTS_DIR/"calibration_plot.png"))
print(cal.round(3))

modeling_table["pos_size"] = evaluation.position_size(
    modeling_table["confidence"], modeling_table["volatility_20d"], max_position_pct=MAX_POSITION_PCT)
filtered = evaluation.confidence_filter(modeling_table, threshold=CONFIDENCE_FLOOR)
stopped = evaluation.stop_loss_rules(modeling_table)
print(f"\nTrades con conf >= {CONFIDENCE_FLOOR}: {(filtered['signal_filtered']!='HOLD').sum()}")
print(f"Stops por sentiment flip:  {stopped['stop_flip'].sum()}")
print(f"Stops por low confidence:  {stopped['stop_low_conf'].sum()}")

## Comparacion A vs B vs C

In [ ]:
comp_models = evaluation.compare_models(
    {"A. Base": bt_base, "B. Sentiment Heavy": bt_sentiment_heavy, "C. Causal HTE": bt_causal},
    benchmark_bt=bh, initial_capital=INITIAL_CAPITAL)
print(comp_models.round(4))
comp_models.to_csv(OUTPUTS_DIR/"model_comparison.csv")

## Signal Card final - se rellena solo

In [ ]:
# Fix para variables perdidas en el restart
GEMINI_MODEL = "gemini-2.5-flash"
n_headlines_modelo = 196
n_signals = len(modeling_table)
n_obs_eval = modeling_table["future_excess_return_5d"].notna().sum()

filas = [
    ("Curso",  "Modelos de IA para Finanzas - EGADE"),
    ("Profesor", "Luis Angel Lozano Medina"),
    ("Equipo", "Mauricio Jazo, Sebastian Aceves, Jose Hernandez"),
    ("Entrega", "08 de junio de 2026"),
    ("Universo", ", ".join(UNIVERSE)),
    ("Benchmark", f"{BENCHMARK}"),
    ("Horizonte", f"{HORIZON} dias habiles"),
    ("Modelo Gemini activo", GEMINI_MODEL),
    ("Headlines en modelo",  str(n_headlines_modelo)),
    ("Obs ticker-fecha total", str(n_signals)),
    ("Modelo A - Total Return", f"{comp_models.loc['A. Base','Total Return']:+.2%}"),
    ("Modelo A - Sharpe",       f"{comp_models.loc['A. Base','Sharpe']:+.2f}"),
    ("Modelo A - Max Drawdown", f"{comp_models.loc['A. Base','Max Drawdown']:+.2%}"),
    ("Modelo C - Total Return", f"{comp_models.loc['C. Causal HTE','Total Return']:+.2%}"),
    ("Modelo C - Sharpe",       f"{comp_models.loc['C. Causal HTE','Sharpe']:+.2f}"),
    ("Buy & Hold JETS Return",  f"{(bh['equity'].iloc[-1]/INITIAL_CAPITAL - 1):+.2%}"),
    ("ROC-AUC",                 f"{auc:.3f}"),
    ("KS statistic",            f"{ks:.3f}"),
    ("Accuracy clasificacion",  f"{metrics_cls['Accuracy']:.1%}"),
    ("F1 Macro",                f"{metrics_cls['F1 Macro']:.3f}"),
]

nl = chr(10)
tabla = "### Signal Card - Sistema Multi-Factor Aerolineas" + nl*2
tabla += "| Campo | Detalle |" + nl + "|---|---|" + nl
tabla += nl.join("| **" + k + "** | " + v + " |" for k, v in filas)

display(Markdown(tabla))
(OUTPUTS_DIR/"signal_card.md").write_text(tabla, encoding="utf-8")
print("\nSignal card guardada.")

## Outputs guardados

In [ ]:
print("Archivos en outputs/:")
for f in sorted(OUTPUTS_DIR.glob("*")):
    print(f"  {f.name:40s} {f.stat().st_size/1024:>7.1f} KB")
print("\nArchivos en data/processed/:")
for f in sorted(DATA_PROC.glob("*")):
    print(f"  {f.name:40s} {f.stat().st_size/1024:>7.1f} KB")
print("\nPipeline completado.")